# SLA-MoE — Full sweep on Colab

This notebook runs the complete revision-ready experiment sweep for the JBHI
resubmission. Designed for the **free Colab T4** GPU; one full run per
artifact scenario fits in a single session.

**Steps**
1. Pick GPU runtime: *Runtime → Change runtime type → T4 GPU*.
2. Clone your fork of the repo.
3. Download EEGdenoiseNet.
4. Run `run_all_experiments.py --mode full`.
5. Download results.zip.

Replace `<your-username>` below.

In [ ]:
# 0. GPU sanity check
!nvidia-smi

In [ ]:
# 1. Clone repo
import os
REPO_URL = 'https://github.com/<your-username>/SLA-MoE.git'   # <-- edit
if not os.path.isdir('/content/SLA-MoE'):
    !git clone $REPO_URL /content/SLA-MoE
%cd /content/SLA-MoE/MoE-main

In [ ]:
# 2. Install deps
!pip install -q torch numpy scipy scikit-learn matplotlib seaborn

In [ ]:
# 3. Download EEGdenoiseNet
import os, urllib.request
os.makedirs('data', exist_ok=True)
BASE = 'https://github.com/ncclabsustech/EEGdenoiseNet/raw/master/data/'
for fname in ['EEG_all_epochs.npy', 'EOG_all_epochs.npy', 'EMG_all_epochs.npy']:
    target = f'data/{fname}'
    if not os.path.exists(target):
        print('Downloading', fname)
        urllib.request.urlretrieve(BASE + fname, target)
!ls -la data/

In [ ]:
# 4a. Smoke test (a few minutes)
!python run_all_experiments.py --mode quick --data-path data

In [ ]:
# 4b. Full sweep — combined EEG+EOG+EMG only first (most important table)
!python run_all_experiments.py --mode full --experiment eeg_eog_emg --data-path data --output-dir results

In [ ]:
# 4c. EEG+EOG and EEG+EMG sweeps (run separately if you hit the session timer)
!python run_all_experiments.py --mode full --experiment eeg_eog --data-path data --output-dir results
!python run_all_experiments.py --mode full --experiment eeg_emg --data-path data --output-dir results

In [ ]:
# 5. Ablation
!python -m src.experiments.ablation --data-path data --output-dir results/ablation

In [ ]:
# 6. Downstream BCI-IV-2a (motor imagery)
!pip install -q moabb mne braindecode
!python -m src.downstream.bci_iv_2a --data-path data --output-dir results/downstream_bci_iv_2a

In [ ]:
# 7. Zip + download results
import shutil
shutil.make_archive('/content/results', 'zip', 'results')
from google.colab import files
files.download('/content/results.zip')